In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

In [3]:
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

In [4]:

url = "https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Master.xml"
driver.get(url)

time.sleep(3)  

html = driver.page_source   
print(html)          

driver.quit()


<html xmlns="http://www.w3.org/1999/xhtml"><head><style id="xml-viewer-style">/* Copyright 2014 The Chromium Authors
 * Use of this source code is governed by a BSD-style license that can be
 * found in the LICENSE file.
 */

:root {
  color-scheme: light dark;
}

div.header {
    border-bottom: 2px solid black;
    padding-bottom: 5px;
    margin: 10px;
}

@media (prefers-color-scheme: dark) {
  div.header {
    border-bottom: 2px solid white;
  }
}

div.folder &gt; div.hidden {
    display:none;
}

div.folder &gt; span.hidden {
    display:none;
}

.pretty-print {
    margin-top: 1em;
    margin-left: 20px;
    font-family: monospace;
    font-size: 13px;
}

#webkit-xml-viewer-source-xml {
    display: none;
}

.opened {
    margin-left: 1em;
}

.comment {
    white-space: pre;
}

.folder-button {
    user-select: none;
    cursor: pointer;
    display: inline-block;
    margin-left: -10px;
    width: 10px;
    background-repeat: no-repeat;
    background-position: left top;
    vert

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

options = Options()
driver = webdriver.Chrome(options=options)

url = "https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Master.xml"
driver.get(url)

time.sleep(2)

html = driver.page_source


In [9]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html)

loc_tags = [tag.text for tag in soup.find_all("loc")]

for loc in loc_tags:
    print(loc)


https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_PartDetail_1_50000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_PartDetail_50001_100000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_CategoryPages.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_1_50000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_50001_100000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_100001_150000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_150001_200000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_200001_250000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_250001_300000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_300001_350000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Models_350001_400000.xml.gz
https://www.partselect.com/sitemaps/PartSelect.com_Sitem

In [10]:
import requests
import os

os.makedirs("sitemaps", exist_ok=True)
sitemap_urls = loc_tags

for url in sitemap_urls:
    filename = url.split("/")[-1]
    path = f"sitemaps/{filename}"

    print("Downloading:", filename)
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

    if resp.status_code == 200:
        with open(path, "wb") as f:
            f.write(resp.content)
    else:
        print("FAILED:", resp.status_code, url)


Downloading: PartSelect.com_Sitemap_PartDetail_1_50000.xml.gz
Downloading: PartSelect.com_Sitemap_PartDetail_50001_100000.xml.gz
Downloading: PartSelect.com_Sitemap_CategoryPages.xml.gz
Downloading: PartSelect.com_Sitemap_Models_1_50000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_50001_100000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_100001_150000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_150001_200000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_200001_250000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_250001_300000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_300001_350000.xml.gz
Downloading: PartSelect.com_Sitemap_Models_350001_400000.xml.gz
Downloading: PartSelect.com_Sitemap_Repairs.xml.gz
Downloading: PartSelect.com_Sitemap_Blogs.xml.gz
Downloading: PartSelect.com_Sitemap_PTLs.xml.gz


In [11]:
import gzip

xml_files = []

for gz_file in os.listdir("sitemaps"):
    if gz_file.endswith(".gz"):
        gz_path = f"sitemaps/{gz_file}"
        xml_path = gz_path.replace(".gz", "")

        with gzip.open(gz_path, "rb") as f_in:
            xml_content = f_in.read()

        with open(xml_path, "wb") as f_out:
            f_out.write(xml_content)

        xml_files.append(xml_path)

print("Decompressed:", len(xml_files), "XML files")


Decompressed: 14 XML files


In [15]:
import xml.etree.ElementTree as ET
from collections import defaultdict

# Dictionary to store URLs by category
urls_by_category = defaultdict(list)

for xml_path in xml_files:
    print("Parsing:", xml_path)

    # Determine category from filename
    fname = xml_path.split("/")[-1].lower()
    if "models" in fname or "partdetail" in fname:
        category = "products"  # club models + parts
    elif "blogs" in fname:
        category = "blogs"
    elif "categorypages" in fname:
        category = "categories"
    elif "ptls" in fname:
        category = "ptls"
    elif "repairs" in fname:
        category = "repairs"
    else:
        category = "other"

    # Parse XML
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}

    # Extract <loc> URLs
    for loc in root.findall(".//sm:loc", ns):
        urls_by_category[category].append(loc.text)

# Example: check counts
for cat, urls in urls_by_category.items():
    print(f"{cat}: {len(urls)} URLs")


Parsing: sitemaps/PartSelect.com_Sitemap_Blogs.xml
Parsing: sitemaps/PartSelect.com_Sitemap_CategoryPages.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_100001_150000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_150001_200000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_1_50000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_200001_250000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_250001_300000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_300001_350000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_350001_400000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Models_50001_100000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_PartDetail_1_50000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_PartDetail_50001_100000.xml
Parsing: sitemaps/PartSelect.com_Sitemap_PTLs.xml
Parsing: sitemaps/PartSelect.com_Sitemap_Repairs.xml
blogs: 210 URLs
categories: 2146 URLs
products: 433516 URLs
ptls: 12661 URLs
repairs: 212 URLs


In [16]:
import csv

for category, urls in urls_by_category.items():
    filename = f"{category}_urls.csv"
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["url"])
        for url in urls:
            writer.writerow([url])
    print(f"Saved {filename} with {len(urls)} URLs")


Saved blogs_urls.csv with 210 URLs
Saved categories_urls.csv with 2146 URLs
Saved products_urls.csv with 433516 URLs
Saved ptls_urls.csv with 12661 URLs
Saved repairs_urls.csv with 212 URLs


In [18]:
import pandas as pd
import glob
import os

# Define the keywords to filter
keywords = ["refrigerator", "dishwasher"]

csv_folder = "./urls"  
csv_files = glob.glob(os.path.join(csv_folder, "*_urls.csv"))

# Folder to save filtered CSVs
filtered_folder = "filtered_urls"
os.makedirs(filtered_folder, exist_ok=True)  

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    
    # Filter URLs that contain any of the keywords
    df_filtered = df[df['url'].str.lower().str.contains("|".join(keywords))]
    
    # Save filtered CSV in the folder
    base_name = os.path.basename(csv_file)
    filtered_name = os.path.join(filtered_folder, base_name)
    df_filtered.to_csv(filtered_name, index=False)
    
    print(f"{base_name}: {len(df_filtered)} URLs saved to {filtered_name}")


blogs_urls.csv: 38 URLs saved to filtered_urls\blogs_urls.csv
categories_urls.csv: 74 URLs saved to filtered_urls\categories_urls.csv
products_urls.csv: 1772 URLs saved to filtered_urls\products_urls.csv
ptls_urls.csv: 1540 URLs saved to filtered_urls\ptls_urls.csv
repairs_urls.csv: 123 URLs saved to filtered_urls\repairs_urls.csv
